# 01 · Inspección de hojas y del punto de operación

## Objetivo
Registrar, para cada hoja visible, el texto de Cluster, el texto de Pozo, la fecha cacheada y el caudal de operación.

## Alcance
Se usa solo `entrada/BES_jun.xlsx`. El lector abre el libro en memoria y no lo guarda. Esta inspección no decide cuál de los dos códigos es la llave del pozo.


In [1]:
from pathlib import Path
import os, sys
aqui = Path.cwd().resolve()
expediente = next(c for c in [aqui, *aqui.parents] if (c / "dataset.md").is_file() and (c / "entrada").is_dir())
os.chdir(expediente)
if str(expediente / "src") not in sys.path:
    sys.path.insert(0, str(expediente / "src"))
libro = expediente / "entrada" / "BES_jun.xlsx"

from ingesta.leer_bes_jun import leer_libro
hojas = leer_libro(libro)
print(f"hojas={len(hojas)}")
for h in hojas:
    print(
        f"hoja={h['hoja']} estado={h['estado_hoja']} cluster={h['cluster']} pozo={h['pozo']} "
        f"fecha_es_today={h['fecha_es_today']} fecha_cache={h['fecha_cache']} "
        f"caudal_operacion_bfpd={h['caudal_operacion_bfpd']} puntos_curva={len(h['curva'])}"
    )
assert len(hojas) == 5
assert all(h["estado_hoja"] == "visible" for h in hojas)
assert all(h["cluster"] == h["hoja"] and h["cluster"] != h["pozo"] for h in hojas)
print("criterio_llave=no_declarada_en_el_libro")


hojas=5
hoja=QF-554H estado=visible cluster=QF-554H pozo=QF-084 fecha_es_today=True fecha_cache=2026-10-17 00:00:00 caudal_operacion_bfpd=10301.0 puntos_curva=6
hoja=QF-545H estado=visible cluster=QF-545H pozo=QF-301 fecha_es_today=True fecha_cache=2026-10-17 00:00:00 caudal_operacion_bfpd=8767.0 puntos_curva=6
hoja=QF-500H estado=visible cluster=QF-500H pozo=QF-147 fecha_es_today=True fecha_cache=2026-10-17 00:00:00 caudal_operacion_bfpd=7728.0 puntos_curva=6
hoja=QF-378HST estado=visible cluster=QF-378HST pozo=QF-005 fecha_es_today=True fecha_cache=2026-10-17 00:00:00 caudal_operacion_bfpd=7116.0 puntos_curva=6
hoja=QF-549HST estado=visible cluster=QF-549HST pozo=QF-306 fecha_es_today=False fecha_cache=2017-09-06 00:00:00 caudal_operacion_bfpd=5419.0 puntos_curva=6
criterio_llave=no_declarada_en_el_libro


## Hechos leídos
Hay cinco hojas y las cinco están visibles. En cada una, el texto junto a Cluster coincide con el nombre de la hoja y el texto junto a Pozo es otro código: QF-084, QF-301, QF-147, QF-005 y QF-306. El libro no dice cuál de los dos identifica el pozo. Por eso la salida termina en `criterio_llave=no_declarada_en_el_libro` y las hojas no se fusionan.

Cuatro hojas tienen `fecha_es_today=True` y fecha cacheada 2026-10-17. Esa fecha es el resultado de `=TODAY()` guardado en el libro, no una fecha de prueba escrita por el usuario. QF-549HST tiene `fecha_es_today=False` y fecha cacheada 2017-09-06. Con un solo punto por hoja no hay serie diaria que ordenar.

Cada hoja trae seis puntos en la tabla de bomba. El caudal de operación cacheado, en BFPD, es 10301, 8767, 7728, 7116 y 5419, en el orden de las hojas impreso arriba.
